In [ ]:
import pandas as pd
from utils import read_json
from core.config import CSV_FILES_PATH

comb5= read_json()


In [ ]:
# file_name='comb5.csv'
# comb5_df  = pd.read_csv(f'{CSV_FILES_PATH}/{file_name}')

In [ ]:
# comb5_df

### Rank, Suit
```python
class Rank(StrEnum):
    ACE = 'A',
    KING = 'K',
    QUEEN = 'Q',
    JACK = 'J',
    _10 = '10',
    _9 = '9',
    _8 = '8',
    _7 = '7',
    _6 = '6',
    _5 = '5',
    _4 = '4',
    _3 = '3',
    _2 = '2',

```
```python
class Suit(StrEnum):
    HEARTS = 'h',
    CLUBS = 'c',
    DIAMONDS = 'd',
    SPADES = 's',
```

### Card
``` python
class Card(BaseModel):
    rank: Rank  
    suit: Suit
    
    @computed_field
    def short_name(self) -> str:
        return f'{self.rank}{self.suit}'
    
    def __hash__(self):
        return hash(self.short_name)
```

In [ ]:
# from subjects import Rank,Suit,Card,Comb5,Deck

# rank = Rank._10
# suit = Suit.CLUBS
# card1 = Card(
#     rank=rank,
#     suit=suit)

# rank = Rank.JACK
# suit = Suit.CLUBS
# card2 = Card(
#     rank=rank,
#     suit=suit)

# rank = Rank.ACE
# suit = Suit.DIAMONDS
# card3 = Card(
#     rank=rank,
#     suit=suit)

# rank = Rank.KING
# suit = Suit.CLUBS
# card4 = Card(
#     rank=rank,
#     suit=suit)

# rank = Rank.ACE
# suit = Suit.CLUBS
# card5 = Card(
#     rank=rank,
#     suit=suit)

# card_set = {card1,card2,card3,card4,card5}


In [ ]:
# card2

### Comb5
```python
class Comb5(BaseModel):
    cards_set: set[Card]
    
    
    @field_validator('cards_set')
    def check_card_amount(cls, value):
        if len(value) != 5:
            raise ValueError('Card amount must be 5')
        return value
    
    @computed_field 
    def flush_is(self)-> bool:
        suit_list = [card.suit for card in self.cards_set]
        return True if len(set(suit_list)) == 1 else False
    
    @computed_field
    def str_key(self) -> str:
        rank_list = [RankTools.rank_to_ind(card.rank)+2 for card in self.cards_set]
        flush = 'fl' if self.flush_is else 'unfl'
        return ','.join(map(str, sorted(rank_list))) + '_' + flush
    
    @computed_field
    def comb_fig(self) -> dict[str,int]:
        name, nn = Comb5Dict.comb5_dict[self.str_key]
        return {'name':name, 
                'nn':nn
                }
```

In [ ]:
# comb5 = Comb5(cards_set=card_set)

In [ ]:
# comb5.comb_fig

### Deck
```python
class Deck(BaseModel):
    cards_set: set[Card]
    
    @computed_field
    def short_view(self) -> dict[Suit,list[str]]:
        deck_view = {
            Suit.HEARTS:[],
            Suit.CLUBS:[],
            Suit.DIAMONDS:[],
            Suit.SPADES:[],
        }
        for curr_card in self.cards_set:
            deck_view[curr_card.suit].append(curr_card.short_name)
        return deck_view
    
    
    
    @computed_field 
    def rank_amount(self)-> list[int]:
        rank_amount = [0 for _ in range(13)]
        for curr_card in self.cards_set:
            rank_amount[RankTools.rank_to_ind(curr_card.rank)] +=1
        return rank_amount
    
    @computed_field 
    def suits_0_1(self)-> list[list[int]]:
        suit_0_1 = [[0 for _ in range(13)] for _ in range(4)]
        for curr_card in self.cards_set:
            suit_0_1[SuitTools.suit_to_ind(curr_card.suit)][RankTools.rank_to_ind(curr_card.rank)] =1
        return suit_0_1
         
    def remove_cards(self, card_list:list[Card]):
        self.cards_set -= set(card_list)
        
    def add_cards(self, card_list:list[Card]):
        self.cards_set |= set(card_list)

    def deal_rnd_cards(self, amount_cards:int)-> list[Card]:
        if len(self.cards_set) < amount_cards:
            raise ValueError('Not enough cards in deck to deal')
        dealt_cards = random.sample(self.cards_set, amount_cards)
        self.remove_cards(dealt_cards)  
        return dealt_cards
```

In [ ]:
# from subjects import Deck,DeckTools

In [ ]:
# full_deck = DeckTools.do_full_deck()

In [ ]:
# full_deck.suits_0_1

In [ ]:
# card_5 = full_deck.deal_rnd_cards(5)

In [ ]:
# full_deck.suits_0_1

In [ ]:
# comb5 = Comb5(cards_set=card_5)

In [ ]:
# comb5.comb_fig